# Deploy DT-Based Calibration Microservices

This notebook deploys the `dt-based-calibration` Docker Compose profile on a GPU-equipped launchable or local Linux host.

The CARLA base image is intentionally not hard-coded here. Set `CARLA_BASE_IMAGE` in Section 1 to an image you are authorized to use. Additional maps are optional; set `ADDITIONAL_MAP_LINK` only when you want the Docker build to import them.

What it does:
1. Validates GPU, Docker, and required configuration
2. Logs in to `nvcr.io` with your NGC API key
3. Locates or clones the VSS deployment repo
4. Writes a dt-based-calibration `generated.env`
5. Builds and starts the CARLA server plus dt-based-calibration service
6. Verifies service ports and prints access URLs

Prerequisites:
- Linux instance with NVIDIA GPU drivers and NVIDIA Container Toolkit
- Docker Engine with Docker Compose v2
- NGC API key from https://ngc.nvidia.com
- An authorized CARLA 0.9.16-compatible base image reference for `CARLA_BASE_IMAGE`
- Optional: a URL for the CARLA additional maps tarball for `ADDITIONAL_MAP_LINK`


## 1. Configuration

Fill in the required values before running the rest of the notebook. `CARLA_BASE_IMAGE` is intentionally blank so the user supplies an authorized CARLA base image themselves.

In [ ]:
import os

# ============================================================
# REQUIRED: Set these before running anything else
# ============================================================

NGC_CLI_API_KEY = ''          # Your NGC API key. Used for nvcr.io docker login.
NGC_CLI_ORG = 'nvidia'        # Optional NGC org. Kept for consistency with other launch notebooks.

# User-provided CARLA base image. Do not commit or publish a restricted image here.
CARLA_BASE_IMAGE = ''

# Optional URL to the CARLA 0.9.16 additional maps archive. Leave empty to skip this long download.
ADDITIONAL_MAP_LINK = ''

# Required by the dt-based-calibration UI/API for map rendering.
GOOGLE_MAPS_API_KEY = ''

# Docker Compose project/profile settings.
COMPOSE_PROJECT_NAME = 'mdx'
DT_COMPOSE_PROFILE = 'dt_based_calibration'

# ============================================================
# OPTIONAL: Override defaults if needed
# ============================================================

# Deployment source. If the current working directory is the repo, it is used.
# Otherwise the default launchable path is tried, then the notebook clones from GitHub.
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, 'deploy', 'docker')):
    _DEFAULT_DEPLOY_SOURCE_PATH = _cwd
else:
    _DEFAULT_DEPLOY_SOURCE_PATH = os.path.expanduser('~/video-search-and-summarization')
DEPLOY_SOURCE_PATH = _DEFAULT_DEPLOY_SOURCE_PATH if os.path.isdir(_DEFAULT_DEPLOY_SOURCE_PATH) else ''

GIT_BRANCH = 'release/3.2.0'

# Network overrides. Leave empty for auto-detection.
HOST_IP_OVERRIDE = ''
EXTERNAL_IP_OVERRIDE = ''


In [ ]:
# ---- Validate configuration ----
from urllib.parse import urlparse

def mask_secret(value, visible=4):
    value = value or ''
    if len(value) <= visible * 2:
        return '***' if value else '<empty>'
    return f'{value[:visible]}...{value[-visible:]}'

assert NGC_CLI_API_KEY, 'NGC_CLI_API_KEY is required. Get one at https://ngc.nvidia.com'
assert CARLA_BASE_IMAGE, 'CARLA_BASE_IMAGE is required. Set it to an authorized CARLA 0.9.16-compatible image.'
if ADDITIONAL_MAP_LINK:
    assert urlparse(ADDITIONAL_MAP_LINK).scheme in ('http', 'https'), 'ADDITIONAL_MAP_LINK must be an http(s) URL.'
assert GOOGLE_MAPS_API_KEY, 'GOOGLE_MAPS_API_KEY is required for map rendering.'
assert DT_COMPOSE_PROFILE == 'dt_based_calibration', f'Unexpected dt compose profile: {DT_COMPOSE_PROFILE}'

if DEPLOY_SOURCE_PATH:
    assert os.path.isdir(DEPLOY_SOURCE_PATH), f'DEPLOY_SOURCE_PATH does not exist: {DEPLOY_SOURCE_PATH}'

# Export values for shell cells and subprocesses.
os.environ['NGC_CLI_API_KEY'] = NGC_CLI_API_KEY
os.environ['CARLA_BASE_IMAGE'] = CARLA_BASE_IMAGE
os.environ['ADDITIONAL_MAP_LINK'] = ADDITIONAL_MAP_LINK
os.environ['GOOGLE_MAPS_API_KEY'] = GOOGLE_MAPS_API_KEY

print('Configuration valid.')
print(f'  Source:              {DEPLOY_SOURCE_PATH or f"GitHub (branch: {GIT_BRANCH})"}')
print(f'  Compose profile:     {DT_COMPOSE_PROFILE}')
print(f'  Compose project:     {COMPOSE_PROJECT_NAME}')
print(f'  NGC org:             {NGC_CLI_ORG or "account default"}')
print(f'  NGC key:             {mask_secret(NGC_CLI_API_KEY)}')
print(f'  CARLA base image:    {CARLA_BASE_IMAGE}')
maps_status = mask_secret(ADDITIONAL_MAP_LINK, visible=12) if ADDITIONAL_MAP_LINK else '<skipped>'
print(f'  Additional maps URL: {maps_status}')
print(f'  Google Maps key:     {mask_secret(GOOGLE_MAPS_API_KEY)}')


## 2. Prerequisites Check

Validate that NVIDIA drivers, Docker, Docker Compose, and the NVIDIA container runtime are available.

In [ ]:
%%bash
set -euo pipefail

MIN_DRIVER_VERSION="550.0.0"
MIN_CUDA_VERSION="12.0"
MIN_DOCKER_VERSION="28.3.3"
MIN_COMPOSE_VERSION="2.39.1"

fail() {
    echo "ERROR: $*" >&2
    exit 1
}

require_command() {
    command -v "$1" >/dev/null 2>&1 || fail "$1 is not installed or not on PATH"
}

version_ge() {
    [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n1)" = "$2" ]
}

echo "=== NVIDIA Driver & GPU ==="
require_command nvidia-smi
nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv,noheader

GPU_COUNT=$(nvidia-smi --query-gpu=index --format=csv,noheader | wc -l | tr -d ' ')
[ "$GPU_COUNT" -gt 0 ] || fail "No NVIDIA GPUs detected"
echo "Detected $GPU_COUNT GPU(s)"

DRIVER_VERSION=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -n1 | tr -d ' ')
version_ge "$DRIVER_VERSION" "$MIN_DRIVER_VERSION" \
    || fail "NVIDIA driver $DRIVER_VERSION is older than required $MIN_DRIVER_VERSION"
echo "NVIDIA driver: OK ($DRIVER_VERSION >= $MIN_DRIVER_VERSION)"

CUDA_VERSION=$(nvidia-smi | awk -F'CUDA Version: ' '/CUDA Version/ {split($2,a," "); version=a[1]} END {print version}')
[ -n "$CUDA_VERSION" ] || fail "Could not read CUDA version from nvidia-smi"
version_ge "$CUDA_VERSION" "$MIN_CUDA_VERSION" \
    || fail "CUDA version $CUDA_VERSION is older than required $MIN_CUDA_VERSION"
echo "CUDA version: OK ($CUDA_VERSION >= $MIN_CUDA_VERSION)"
echo ""

echo "=== Docker ==="
require_command docker
docker ps >/dev/null 2>&1 \
    || fail "Docker daemon is not reachable by this user. Add the user to the docker group or start Docker."

DOCKER_VERSION=$(docker version --format '{{.Server.Version}}')
version_ge "$DOCKER_VERSION" "$MIN_DOCKER_VERSION" \
    || fail "Docker Engine $DOCKER_VERSION is older than required $MIN_DOCKER_VERSION"
echo "Docker Engine: OK ($DOCKER_VERSION >= $MIN_DOCKER_VERSION)"

COMPOSE_VERSION=$(docker compose version --short 2>/dev/null | sed 's/^v//')
[ -n "$COMPOSE_VERSION" ] || fail "Docker Compose plugin is not installed"
version_ge "$COMPOSE_VERSION" "$MIN_COMPOSE_VERSION" \
    || fail "Docker Compose $COMPOSE_VERSION is older than required $MIN_COMPOSE_VERSION"
echo "Docker Compose: OK ($COMPOSE_VERSION >= $MIN_COMPOSE_VERSION)"
echo ""

echo "=== NVIDIA Container Toolkit ==="
docker run --rm --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi >/dev/null 2>&1 \
    || fail "NVIDIA Container Toolkit is not functional. Install or repair it."
echo "NVIDIA Container Toolkit: OK"
echo ""

echo "=== Disk Space ==="
df -h / | tail -1 | awk '{print "Root:", $4, "available of", $2}'
echo "Prerequisites check passed."


## 3. Locate Deployment Code

Use the local repo if available. Otherwise clone the selected branch from GitHub.

In [ ]:
import os
import subprocess
from pathlib import Path

def run_cmd(cmd, cwd=None):
    print('+ ' + ' '.join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}\n{result.stderr}\n{result.stdout}')
    return result.stdout.strip()

if DEPLOY_SOURCE_PATH:
    REPO_DIR = Path(DEPLOY_SOURCE_PATH).expanduser().resolve()
else:
    REPO_DIR = Path('~/video-search-and-summarization').expanduser().resolve()
    if not REPO_DIR.exists():
        run_cmd([
            'git', 'clone', '--depth', '1', '--branch', GIT_BRANCH,
            'https://github.com/NVIDIA-AI-Blueprints/video-search-and-summarization.git',
            str(REPO_DIR),
        ])

DOCKER_DIR = REPO_DIR / 'deploy' / 'docker'
DT_DIR = DOCKER_DIR / 'industry-profiles' / 'dt-based-calibration'
DT_COMPOSE = DT_DIR / 'compose.yml'
DT_DOCKERFILE = DT_DIR / 'Dockerfile'

assert DOCKER_DIR.is_dir(), f'deploy/docker not found in {REPO_DIR}'
assert DT_COMPOSE.is_file(), f'DT compose file not found: {DT_COMPOSE}'
assert DT_DOCKERFILE.is_file(), f'DT Dockerfile not found: {DT_DOCKERFILE}'

print(f'Repo:       {REPO_DIR}')
print(f'Docker dir: {DOCKER_DIR}')
print(f'DT dir:     {DT_DIR}')


## 4. Network Detection

Detect internal and external IPs for URL printing. Set overrides in Section 1 if auto-detection is not correct.

In [ ]:
import os
import subprocess

def read_etc_environment():
    env = {}
    try:
        with open('/etc/environment') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                key, value = line.split('=', 1)
                env[key.strip()] = value.strip().strip('"').strip("'")
    except FileNotFoundError:
        pass
    return env

def detect_host_ip():
    try:
        result = subprocess.run(
            ['bash', '-lc', "ip route get 1.1.1.1 | awk '/src/ {for (i=1;i<=NF;i++) if ($i==\"src\") print $(i+1)}'"],
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip().splitlines()[0]
    except Exception:
        return '127.0.0.1'

HOST_IP = HOST_IP_OVERRIDE or detect_host_ip()
EXTERNAL_IP = EXTERNAL_IP_OVERRIDE or HOST_IP

_etc_env = read_etc_environment()
BREV_ENV_ID = os.environ.get('BREV_ENV_ID') or _etc_env.get('BREV_ENV_ID', '')
if BREV_ENV_ID:
    os.environ['BREV_ENV_ID'] = BREV_ENV_ID

print(f'HOST_IP:     {HOST_IP}')
print(f'EXTERNAL_IP: {EXTERNAL_IP}')
print(f'BREV_ENV_ID: {BREV_ENV_ID or "<not detected>"}')


## 5. Generate dt-based-calibration Env

Write a local `generated.env` next to the dt compose file. This file feeds Docker Compose interpolation and Docker build args.

In [ ]:
from pathlib import Path

DT_ENV = DT_DIR / 'generated.env'

def dotenv_quote(value):
    value = str(value)
    value = value.replace('\\', '\\\\').replace('"', '\\"')
    return f'"{value}"'

env_values = {
    'COMPOSE_PROJECT_NAME': COMPOSE_PROJECT_NAME,
    'COMPOSE_PROFILES': DT_COMPOSE_PROFILE,
    'CARLA_BASE_IMAGE': CARLA_BASE_IMAGE,
    'ADDITIONAL_MAP_LINK': ADDITIONAL_MAP_LINK,
    'GOOGLE_MAPS_API_KEY': GOOGLE_MAPS_API_KEY,
    'HOST_IP': HOST_IP,
    'EXTERNAL_IP': EXTERNAL_IP,
}

lines = [
    '# Generated by deploy_dt_based_calibration_launchable.ipynb',
    '# Do not commit secrets or restricted image references from this file.',
]
for key, value in env_values.items():
    lines.append(f'{key}={dotenv_quote(value)}')

DT_ENV.write_text('\n'.join(lines) + '\n')

for key, value in env_values.items():
    os.environ[key] = str(value)

print(f'Wrote {DT_ENV}')
print('Keys written:')
for key in env_values:
    suffix = ' (masked)' if key in {'ADDITIONAL_MAP_LINK', 'GOOGLE_MAPS_API_KEY'} else ''
    print(f'  {key}{suffix}')


## 6. Docker Registry Login

Log in to `nvcr.io` so Docker Compose can pull the dt-based-calibration image. If your `CARLA_BASE_IMAGE` is in another private registry, log in to that registry separately before the deploy cell.

In [ ]:
import subprocess

print('Logging in to nvcr.io...')
result = subprocess.run(
    ['docker', 'login', 'nvcr.io', '--username', '$oauthtoken', '--password-stdin'],
    input=NGC_CLI_API_KEY,
    text=True,
    capture_output=True,
)
if result.returncode != 0:
    raise RuntimeError(f'nvcr.io login failed:\n{result.stderr}\n{result.stdout}')
print('nvcr.io login complete.')


## 7. Deploy

Build the CARLA-derived image with the notebook-provided build args, then start the CARLA server and dt-based-calibration service.

In [ ]:
import subprocess
import time

cmd = [
    'docker', 'compose',
    '--env-file', str(DT_ENV),
    '-f', str(DT_COMPOSE),
    '-p', COMPOSE_PROJECT_NAME,
    '--profile', DT_COMPOSE_PROFILE,
    'up', '--detach', '--force-recreate', '--build',
]

print('+ ' + ' '.join(cmd))
process = subprocess.Popen(
    cmd,
    cwd=str(DOCKER_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')

process.wait()
if process.returncode != 0:
    raise RuntimeError(f'Docker Compose deployment failed with exit code {process.returncode}')

print('\nDeployment command completed.')


## 8. Verify Deployment

Poll the exposed service ports. CARLA can take a few minutes to initialize.

In [ ]:
import socket
import subprocess
import time

print('=== Containers ===')
subprocess.run(['docker', 'ps', '--format', 'table {{.Names}}\t{{.Status}}\t{{.Ports}}'])
print()

checks = [
    ('CARLA RPC', 'localhost', 2000),
    ('ROI Rectifier', 'localhost', 8080),
    ('DT API', 'localhost', 7865),
    ('Accuracy API', 'localhost', 8000),
    ('Gradio Demo', 'localhost', 7860),
    ('Gradio Manual', 'localhost', 7861),
]

def port_open(host, port, timeout=3):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

MAX_RETRIES = 30
RETRY_INTERVAL = 10
pending = list(checks)
results = {}

for attempt in range(1, MAX_RETRIES + 1):
    still_pending = []
    for name, host, port in pending:
        if port_open(host, port):
            results[name] = f'OK ({port})'
        else:
            still_pending.append((name, host, port))
    pending = still_pending
    if not pending:
        break
    waiting = ', '.join(name for name, _, _ in pending)
    print(f'[{attempt}/{MAX_RETRIES}] Waiting for: {waiting}')
    time.sleep(RETRY_INTERVAL)

for name, _, port in pending:
    results[name] = f'FAILED ({port})'

print()
all_ok = True
for name, status in results.items():
    if status.startswith('FAILED'):
        all_ok = False
    print(f'  {name:.<24s} {status}')

print()
if all_ok:
    print('All dt-based-calibration ports are reachable.')
else:
    print('Some ports are not reachable yet. Check logs with:')
    print(f'  docker compose --env-file {DT_ENV} -f {DT_COMPOSE} -p {COMPOSE_PROJECT_NAME} logs')


## 9. Access URLs

Use these URLs from your browser. On Brev, create secure links for the ports you need to open.

In [ ]:
services = [
    ('ROI Rectifier', 8080),
    ('DT API', 7865),
    ('Accuracy API', 8000),
    ('Gradio Demo', 7860),
    ('Gradio Manual', 7861),
]

if BREV_ENV_ID:
    print('Brev secure-link URLs:')
    for name, port in services:
        print(f'  {name:.<18s} https://{port}-{BREV_ENV_ID}.brevlab.com')
    print('\nCreate a Brev secure link for each port you need to access.')
else:
    host = EXTERNAL_IP or HOST_IP
    print('Direct URLs:')
    for name, port in services:
        print(f'  {name:.<18s} http://{host}:{port}')
    print('\nIf direct access is blocked, use SSH port forwarding. Example:')
    print(f'  ssh -L 7860:localhost:7860 <user>@{host}')


## 10. Stop Deployment

Stop containers without deleting images or other Docker state.

In [ ]:
import subprocess

cmd = [
    'docker', 'compose',
    '--env-file', str(DT_ENV),
    '-f', str(DT_COMPOSE),
    '-p', COMPOSE_PROJECT_NAME,
    '--profile', DT_COMPOSE_PROFILE,
    'stop',
]
print('+ ' + ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(DOCKER_DIR), capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Stop failed with exit code {result.returncode}')
print('dt-based-calibration containers stopped.')


## 11. Teardown

Stop and remove the dt-based-calibration compose resources for this project.

In [ ]:
import subprocess

cmd = [
    'docker', 'compose',
    '--env-file', str(DT_ENV),
    '-f', str(DT_COMPOSE),
    '-p', COMPOSE_PROJECT_NAME,
    '--profile', DT_COMPOSE_PROFILE,
    'down', '--volumes', '--remove-orphans',
]
print('+ ' + ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(DOCKER_DIR), capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Teardown failed with exit code {result.returncode}')
print('dt-based-calibration compose resources removed.')
